# Лабораторная работа №5

## Сжатие модели и оптимизация инференса


### Цель

Построить воспроизводимый конвейер оптимизации инференса модели компьютерного зрения и исследовать компромисс между качеством, задержкой, пропускной способностью, размером модели и потреблением памяти.

Результатом работы является не максимально сжатая модель, а обоснованный выбор конфигурации для заданного сценария эксплуатации.

## 1. Что используется в работе

Преподаватель предоставляет:

- компактный размеченный датасет;
- готовую обученную модель классификации или детекции;
- фиксированный test-набор;
- подготовленное окружение с `PyTorch`, `ONNX Runtime`;
- при наличии GPU — TensorRT или `torch.compile`;
- ограничение на вычислительный бюджет.

Обязательная часть включает:

1. измерение baseline;
2. экспорт модели;
3. статическое или динамическое квантование;
4. сравнение качества и ресурсоёмкости;
5. одно дополнительное исследование.

Полное переобучение модели не входит в обязательную часть.

## 2. Краткая теоретическая справка

### 2.1. Что означает оптимизация инференса

Оптимизация может уменьшать:

- число операций;
- объём памяти;
- размер весов;
- задержку;
- энергопотребление.

При этом возможна деградация качества и изменение поведения модели на отдельных классах.

### 2.2. Квантование

При квантовании веса и, иногда, активации представляются числами меньшей разрядности.

- **Dynamic quantization** обычно применяется к весам.
- **Post-training static quantization** требует calibration-набора.
- **Quantization-aware training** моделирует квантование во время обучения.

### 2.3. Прунинг

Прунинг удаляет отдельные веса, каналы или блоки. Неструктурированный прунинг уменьшает число ненулевых весов, но не всегда ускоряет реальный инференс. Структурированный прунинг потенциально даёт ускорение, но сильнее изменяет архитектуру.

### 2.4. Дистилляция

Student-модель обучается воспроизводить поведение teacher-модели. Это способ перенести знания в более компактную архитектуру, но он требует дополнительного обучения и не входит в обязательное ядро этой работы.

### 2.5. Корректное измерение

Измерение latency должно включать:

- прогрев;
- одинаковый batch size;
- одинаковую предобработку;
- синхронизацию GPU;
- многократные повторения;
- mean, median и p95.

Первый запуск и время загрузки модели не смешиваются с устойчивой latency инференса.

## 3. Задачи

1. Проверить test-набор и baseline-модель.
2. Измерить качество baseline.
3. Измерить:
   - размер файла;
   - число параметров;
   - latency;
   - throughput;
   - пиковую память.
4. Экспортировать модель в ONNX.
5. Проверить численную и функциональную эквивалентность.
6. Выполнить квантование.
7. Сравнить baseline, ONNX и quantized-модель.
8. Провести одно дополнительное исследование:
   - FP16;
   - dynamic против static INT8;
   - влияние batch size;
   - структурированный прунинг;
   - `torch.compile`;
   - иной согласованный фактор.
9. Проанализировать деградацию по классам и типам входов.
10. Выбрать конфигурацию под заданный эксплуатационный сценарий.

## 4. Подготовка данных и сценария эксплуатации

Используйте фиксированный test-набор. Для static quantization отдельно сформируйте calibration-набор из train/validation данных.

До оптимизации зафиксируйте сценарий:

- устройство: CPU или GPU;
- batch size;
- максимальная допустимая latency;
- минимально допустимое качество;
- ограничение по памяти или размеру модели.

Пример сценария:

> CPU-инференс, batch size 1, p95 latency не более 50 мс, снижение macro F1 не более 1 п.п.

In [ ]:
from pathlib import Path
import json
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

try:
    import onnxruntime as ort
except ImportError:
    ort = None

DATA_ROOT = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("ONNX Runtime available:", ort is not None)

In [ ]:
# TODO: загрузите test manifest.
# Минимальные столбцы:
# path, label, split

dataset_table = pd.DataFrame(columns=["path", "label", "split"])
dataset_table.head()

In [ ]:
# TODO: сформулируйте эксплуатационный сценарий.

deployment_scenario = {
    "device": "cpu",
    "batch_size": 1,
    "max_p95_latency_ms": None,
    "min_quality": None,
    "max_model_size_mb": None,
}

deployment_scenario

In [ ]:
def validate_dataset(table: pd.DataFrame) -> None:
    pass

def build_test_loader(table: pd.DataFrame, batch_size: int):
    pass


**Контрольная точка 1**

До оптимизации должны быть готовы:

- проверенный test-набор;
- calibration-набор при static quantization;
- зафиксированный сценарий эксплуатации;
- единая предобработка;
- единый набор метрик.

## 5. Baseline и измерительный стенд

Baseline должен измеряться тем же кодом, что и оптимизированные варианты.

Минимальные метрики качества:

- accuracy;
- macro F1;
- per-class recall;
- confusion matrix.

Минимальные системные показатели:

- mean latency;
- median latency;
- p95 latency;
- throughput;
- peak memory;
- размер модели.

In [ ]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 2)

@torch.no_grad()
def evaluate_torch_model(
    model: nn.Module,
    loader,
    device: torch.device,
) -> dict:
    """Вернуть метрики качества и предсказания."""
    pass


In [ ]:
def benchmark_callable(
    inference_fn,
    sample_input,
    *,
    warmup: int = 20,
    repeats: int = 100,
    synchronize=None,
) -> dict:
    """Измерить mean, median, p95 latency и throughput."""
    pass

In [ ]:
def measure_peak_memory_torch(
    model: nn.Module,
    sample_input: torch.Tensor,
    device: torch.device,
) -> float:
    """Вернуть пиковую память в MB."""
    pass


## 6. Экспорт в ONNX

Экспорт должен использовать фиксированную сигнатуру входа и корректный режим `eval`.

После экспорта сравните выходы PyTorch и ONNX Runtime на нескольких батчах. Небольшое численное различие допустимо, но предсказанные классы и метрики должны оставаться согласованными.

In [ ]:
def export_to_onnx(
    model: nn.Module,
    sample_input: torch.Tensor,
    output_path: Path,
    *,
    dynamic_batch: bool = True,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dynamic_axes = None
    if dynamic_batch:
        dynamic_axes = {
            "input": {0: "batch"},
            "output": {0: "batch"},
        }

    pass

In [ ]:
def create_onnx_session(model_path: Path, provider: str = "CPUExecutionProvider"):
    if ort is None:
        raise RuntimeError("Установите onnxruntime.")
    pass

def validate_onnx_equivalence(
    torch_model: nn.Module,
    onnx_session,
    sample_batches: list,
    device: torch.device,
) -> pd.DataFrame:
    """Сравнить численные выходы и предсказания."""
    pass


## 7. Квантование

Обязательная конфигурация — INT8 post-training quantization.

Допускаются два варианта:

- dynamic quantization, если архитектура и runtime поддерживают;
- static quantization с calibration-набором.

В отчёте явно укажите:

- что именно квантовано;
- тип quantization;
- calibration protocol;
- backend;
- неподдерживаемые операции.

In [ ]:
@dataclass(frozen=True)
class OptimizationConfig:
    name: str
    backend: str
    precision: str
    batch_size: int
    quantization_type: str | None = None
    calibration_size: int | None = None
    pruning_ratio: float | None = None
    compile_mode: str | None = None

required_configs = [
    OptimizationConfig(
        name="pytorch_fp32",
        backend="pytorch",
        precision="fp32",
        batch_size=1,
    ),
    OptimizationConfig(
        name="onnx_fp32",
        backend="onnxruntime",
        precision="fp32",
        batch_size=1,
    ),
    OptimizationConfig(
        name="onnx_int8",
        backend="onnxruntime",
        precision="int8",
        quantization_type="static",
        calibration_size=128,
        batch_size=1,
    ),
]

required_configs

In [ ]:
def quantize_onnx_model(
    input_model_path: Path,
    output_model_path: Path,
    *,
    quantization_type: str,
    calibration_loader=None,
) -> None:
    """Создать INT8 ONNX-модель."""
    pass


## 8. Экспериментальный конвейер

Каждый запуск должен сохранять:

- backend;
- precision;
- batch size;
- тип quantization;
- размер calibration-набора;
- метрики качества;
- latency mean/median/p95;
- throughput;
- peak memory;
- размер модели;
- provider;
- версии библиотек;
- статус и ошибку.

In [ ]:
RUNS_PATH = OUTPUT_DIR / "runs.jsonl"

def append_jsonl(path: Path, record: dict) -> None:
    pass

def run_optimization_experiment(
    config: OptimizationConfig,
    test_loader,
    sample_input,
) -> dict:
    pass


**Контрольная точка 2**

Конвейер считается готовым, если:

- все конфигурации используют один test-набор;
- preprocessing идентичен;
- warmup и repeats фиксированы;
- GPU синхронизируется;
- качество и latency сохраняются в одном журнале;
- повторный запуск не дублирует завершённые серии.

## 9. Дополнительное исследование

Выберите один фактор:

- batch size;
- FP16;
- dynamic против static INT8;
- размер calibration-набора;
- structured pruning;
- `torch.compile`;
- иной согласованный фактор.

Дополнительная серия ограничена **2–4 конфигурациями**.

Сформулируйте исследовательский вопрос и гипотезу до запуска.

In [ ]:
research_question = ""
hypothesis = ""
extra_configs = []

## 10. Оценка, анализ и сдача

Обязательное сравнение:

1. PyTorch FP32;
2. ONNX Runtime FP32;
3. ONNX Runtime INT8;
4. дополнительная серия.

Постройте:

- quality vs p95 latency;
- quality vs model size;
- throughput по конфигурациям;
- peak memory;
- деградацию per-class recall;
- Pareto frontier допустимых решений.

In [ ]:
results = pd.DataFrame()
results

In [ ]:
def summarize_results(results: pd.DataFrame) -> pd.DataFrame:
    pass

def mark_feasible_configs(
    results: pd.DataFrame,
    scenario: dict,
) -> pd.DataFrame:
    """Отметить конфигурации, удовлетворяющие ограничениям сценария."""
    pass


In [ ]:
# TODO: построить отдельные графики:
# 1. quality vs p95 latency;
# 2. quality vs model size;
# 3. throughput;
# 4. peak memory;
# 5. per-class degradation.


### Анализ деградации

Выберите:

- не менее трёх классов без заметной деградации;
- не менее трёх классов с максимальной деградацией;
- не менее пяти изображений, где предсказание изменилось после квантования.

Для каждого случая укажите:

- исходные logits или confidence;
- предсказания baseline и optimized;
- возможную причину чувствительности;
- связь с низкой уверенностью, классом, контрастом или входным распределением.

### Обязательные артефакты

1. Проверенный test-набор.
2. Эксплуатационный сценарий.
3. Baseline quality report.
4. Baseline benchmark.
5. ONNX-модель.
6. Проверка эквивалентности.
7. INT8-модель.
8. Единый журнал экспериментов.
9. Дополнительное исследование.
10. Сводная таблица.
11. Не менее пяти графиков.
12. Анализ деградации.
13. Выбор конфигурации под сценарий.
14. Итоговые выводы.


## Критерии оценивания

- Подготовка и сценарий
- Baseline 
- Экспорт ONNX
- Квантование
- Измерительный конвейер
- Основное сравнение
- Дополнительное исследование
- Представление результатов
- Анализ и выводы

### Условия зачёта

Работа не засчитывается, если:

- разные конфигурации измерялись на разных данных;
- отсутствовал warmup;
- GPU latency измерялась без синхронизации;
- test использовался для подбора calibration или параметров;
- приведён только лучший результат без полного сравнения.